<a href="https://colab.research.google.com/github/mohanrana1/My-Agentic-AI-Journey/blob/main/Reflect_basic_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

In [ ]:
import os
from groq import Groq
from google.colab import userdata
api= userdata.get('groq_apikey')

# ================== SETUP ==================
os.environ["GROQ_API_KEY"] = api
client = Groq()

def llm_call(prompt: str, temperature=0.7):
    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [{"role":"user", "content": prompt}],
        temperature = temperature,
        max_tokens= 1200
    )
    return response.choices[0].message.content.strip()

# ================== REFLECTION AGENT ==================
def generate_initial_draft(task: str):
    prompt = f"""
    You are a helpful assitant. Complete the following task as best as you can:

    Task: {task}

    Provide a clear and complete response.
    """
    draft = llm_call(prompt)
    print("Initial Draft Generated\n")
    return draft

def critique(draft: str, task: str):
    prompt= f"""
    You are a strict critic and expert reviewer.

    Task: {task}

    Here is the current draft:
    {draft}

    Carefully analyze the draft and give constructive criticism.
    Point out:
    - What is missing or incomplete
    - What is unclear or poorly written
    - Any factual errors or weak reasoning
    - How it can be improved

    Be specific and honest. Do NOT rewrite it yet, just give feedback.
    """
    feedback = llm_call(prompt, temperature=0.5)
    print("🔍 Critique / Feedback:\n", feedback[:500] + "..." if len(feedback) > 500 else feedback, "\n")
    return feedback

def improve_draft(draft: str, feedback: str, task: str):
    prompt= f"""
    You are an expert improver.

    Original Task: {task}

    Previous Draft:
    {draft}

    Feedback from critic:
    {feedback}

    Now rewrite and improve the draft significantly based on the feedback.
    Make it clearer, more complete, accurate, and professional.
    """
    improved = llm_call(prompt, temperature=0.6)
    return improved


# ================== MAIN REFLECTION LOOP ==================

def reflection_agent(task: str, max_iterations=3):
    print(f"🚀 Starting Reflection Agent for task:\n{task}\n")

    current_draft = generate_initial_draft(task)

    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")

        feedback = critique(current_draft, task)
        current_draft = improve_draft(current_draft,feedback, task)

        print("Improved Version:\n", current_draft[:600] + "..." if len(current_draft) > 600 else current_draft)

        # Optional: Ask if you want to stop early
        if i < max_iterations - 1:
            cont = input("Do you want to do another reflection round? (y/n): ").strip().lower()
            if cont != 'y':
                break

    print("\n" + "="*70)
    print("✅ FINAL IMPROVED OUTPUT AFTER REFLECTION")
    print("="*70)
    print(current_draft)
    return current_draft

# ================== RUN IT ==================
if __name__ == "__main__":
    user_task = "Write a detailed guide on how to start freelancing in AI Agent development in 2026, especially from Nepal."
    reflection_agent(user_task, max_iterations=3)